# Tutorial INIAMET - Datos Agroclimáticos de Chile

Librería Python para acceder a datos de estaciones agrometeorológicas de INIA Chile.

**Contenido:**
1. Conexión
2. Listar estaciones
3. Variables disponibles
4. Descargar datos (15 min, horario, diario, mensual)
5. Descarga masiva (múltiples estaciones)
6. Exportar datos
7. Gráfico rápido
8. Caché automático

---
## 1. Conexión

```bash
pip install -e .
```

Necesitas una API key de [agromet.inia.cl](https://agromet.inia.cl/api/v2/).

In [ ]:
from iniamet import INIAClient

client = INIAClient(api_key="6c66ed32deb984ce0fb41ffdc54c1e98b517fa50")
print("Conectado ✓")

---
## 2. Listar estaciones

Filtra por región usando el código con prefijo **R**:

| Código | Región |
|---|---|
| R15 | Arica y Parinacota |
| R01 | Tarapacá |
| R02 | Antofagasta |
| R03 | Atacama |
| R04 | Coquimbo |
| R05 | Valparaíso |
| R13 | Metropolitana |
| R06 | O'Higgins |
| R07 | Maule |
| R16 | Ñuble |
| R08 | Biobío |
| R09 | La Araucanía |
| R14 | Los Ríos |
| R10 | Los Lagos |
| R11 | Aysén |
| R12 | Magallanes |

In [ ]:
# Todas las estaciones del país
todas = client.get_stations()
print(f"Total estaciones en Chile: {len(todas)}")
todas.head()

In [ ]:
# Filtrar por región (acepta nombre, código o número)
nuble = client.get_stations(region="Ñuble")     # por nombre
# nuble = client.get_stations(region="R16")     # por código
print(f"Estaciones en Ñuble: {len(nuble)}")
nuble[['codigo', 'nombre', 'comuna']].head(10)

In [ ]:
# Varias regiones a la vez
centro_sur = client.get_stations(region=["R16", "R08", "R07"])
print(f"Estaciones Ñuble + Biobío + Maule: {len(centro_sur)}")
centro_sur.groupby('region')['codigo'].count()

In [ ]:
# Filtrar por tipo de estación
inia_nuble = client.get_stations(region="R16", station_type="INIA")
print(f"Estaciones INIA en Ñuble: {len(inia_nuble)}")
inia_nuble[['codigo', 'nombre', 'comuna']]

---
## 3. Variables disponibles

Cada estación mide distintas variables. La librería incluye constantes para las más comunes:

| Constante | ID | Variable |
|---|---|---|
| `VAR_TEMPERATURA_MEDIA` | 2002 | Temperatura del aire (°C) |
| `VAR_PRECIPITACION` | 2001 | Precipitación (mm) |
| `VAR_HUMEDAD_RELATIVA` | 2007 | Humedad relativa (%) |
| `VAR_RADIACION_MEDIA` | 2022 | Radiación solar (W/m²) |
| `VAR_VIENTO_VELOCIDAD_MEDIA` | 2013 | Velocidad viento (m/s) |
| `VAR_VIENTO_VELOCIDAD_MAXIMA` | 2014 | Ráfaga máxima (m/s) |
| `VAR_VIENTO_DIRECCION` | 2012 | Dirección viento (°) |
| `VAR_PRESION_ATMOSFERICA` | 2125 | Presión (mbar) |

In [ ]:
# Ver qué variables tiene una estación específica
variables = client.get_variables("INIA-47")  # Ninhue, Ñuble
variables

In [ ]:
# Tabla completa de variables conocidas
from iniamet import list_all_variables
list_all_variables()

---
## 4. Descargar datos de UNA estación

### Agregaciones disponibles

| Alias español | Alias inglés | Resolución |
|---|---|---|
| _(nada)_ / `"raw"` | — | Cada 15 minutos |
| `"horario"` | `"hourly"`, `"H"` | Horario |
| `"diario"` | `"daily"`, `"D"` | Diario |
| `"semanal"` | `"weekly"`, `"W"` | Semanal |
| `"mensual"` | `"monthly"`, `"M"` | Mensual |

### 4a. Datos crudos (cada 15 minutos)

In [ ]:
from iniamet import VAR_TEMPERATURA_MEDIA, VAR_PRECIPITACION

# Temperatura cada 15 minutos
temp_raw = client.get_data(
    station="INIA-47",
    variable=VAR_TEMPERATURA_MEDIA,
    start_date="2025-01-01",
    end_date="2025-01-31"
)

print(f"Registros: {len(temp_raw)}")
print(f"Columnas: {list(temp_raw.columns)}")
temp_raw.head()

### 4b. Agregación horaria

In [ ]:
temp_horario = client.get_data(
    "INIA-47", VAR_TEMPERATURA_MEDIA,
    "2025-01-01", "2025-01-31",
    aggregation="horario"
)
print(f"Registros horarios: {len(temp_horario)}")
temp_horario.head()

### 4c. Agregación diaria

Para **temperatura** calcula automáticamente min, max y media:

In [ ]:
temp_diario = client.get_data(
    "INIA-47", VAR_TEMPERATURA_MEDIA,
    "2025-01-01", "2025-01-31",
    aggregation="diario"
)
print(f"Días: {len(temp_diario)}")
print(f"Columnas: {list(temp_diario.columns)}")
temp_diario.head()

### 4d. Precipitación diaria (se suma, no se promedia)

In [ ]:
pp_diario = client.get_data(
    "INIA-47", VAR_PRECIPITACION,
    "2025-06-01", "2025-06-30",
    aggregation="diario"
)
print(f"Días con datos: {len(pp_diario)}")
pp_diario.head()

### 4e. Agregación mensual

In [ ]:
temp_mensual = client.get_data(
    "INIA-47", VAR_TEMPERATURA_MEDIA,
    "2025-01-01", "2025-12-31",
    aggregation="mensual"
)
temp_mensual

---
## 5. Descarga masiva (múltiples estaciones)

### 5a. Estaciones específicas

In [ ]:
from iniamet import VAR_HUMEDAD_RELATIVA

datos = client.bulk_download(
    stations=["INIA-47", "INIA-139", "INIA-351"],
    variables=[VAR_TEMPERATURA_MEDIA, VAR_HUMEDAD_RELATIVA],
    start_date="2025-01-01",
    end_date="2025-01-31"
)

# Resultado: diccionario {"estacion_variable": DataFrame}
for key, df in datos.items():
    print(f"{key}: {len(df)} registros")

### 5b. Todas las estaciones de una región

In [ ]:
estaciones_nuble = client.get_stations(region="Ñuble")
codigos = estaciones_nuble['codigo'].tolist()
print(f"Descargando temperatura de {len(codigos)} estaciones...")

datos_region = client.bulk_download(
    stations=codigos,
    variables=[VAR_TEMPERATURA_MEDIA],
    start_date="2025-01-01",
    end_date="2025-01-07"
)

print(f"\nEstaciones con datos: {len(datos_region)}")
for key, df in datos_region.items():
    print(f"  {key}: {len(df)} registros")

---
## 6. Exportar datos

Todos los datos se devuelven como **pandas DataFrame**:

In [ ]:
df = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-07")

print(f"Tipo:     {type(df).__name__}")
print(f"Columnas: {list(df.columns)}")
print(f"Filas:    {len(df)}")
print()
print(df.dtypes)

In [ ]:
# Exportar a CSV
df.to_csv("temperatura_inia47.csv", index=False)

# Exportar a Excel (requiere openpyxl)
# df.to_excel("temperatura_inia47.xlsx", index=False)

print("Archivo guardado ✓")

---
## 7. Gráfico rápido

In [ ]:
import matplotlib.pyplot as plt

td = client.get_data(
    "INIA-47", VAR_TEMPERATURA_MEDIA,
    "2025-01-01", "2025-03-31",
    aggregation="diario"
)

plt.figure(figsize=(12, 4))
plt.fill_between(td['tiempo'], td['valor_min'], td['valor_max'], alpha=0.3, label='Min-Max')
plt.plot(td['tiempo'], td['valor'], 'r-', lw=1, label='Media')
plt.ylabel('Temperatura (°C)')
plt.title('INIA-47 Ninhue — Temperatura diaria (ene-mar 2025)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Caché automático

Los datos se guardan en `iniamet_cache/`. La segunda vez se leen del disco (instantáneo).

- Desactivar: `INIAClient(cache=False)`
- Limpiar: `client.cache_manager.clear_cache()`
- Cambiar ruta: `INIAClient(cache_dir="mi_cache")`

In [ ]:
import time

t0 = time.time()
df1 = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-02-01", "2025-02-28")
print(f"Primera descarga: {time.time() - t0:.1f}s ({len(df1)} registros)")

t0 = time.time()
df2 = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-02-01", "2025-02-28")
print(f"Desde caché:      {time.time() - t0:.3f}s ({len(df2)} registros)")

---
## Resumen de la API

```python
from iniamet import INIAClient, VAR_TEMPERATURA_MEDIA, VAR_PRECIPITACION

client = INIAClient(api_key="...")             # Conectar

# Estaciones
client.get_stations()                           # Todas
client.get_stations(region="Ñuble")             # Una región
client.get_stations(region=["R16", "R08"])      # Varias regiones

# Variables
client.get_variables("INIA-47")

# Datos (una estación)
client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-12-31")
client.get_data(..., aggregation="diario")      # horario, diario, semanal, mensual

# Datos (múltiples estaciones)
client.bulk_download(stations=[...], variables=[...], start_date=..., end_date=...)
```

# Tutorial INIAMET - Datos Agroclimáticos de Chile

Librería para acceder a datos de estaciones agrometeorológicas de INIA Chile.

**Contenido:**
1. Conexión
2. Listar estaciones (1 región o varias)
3. Variables disponibles
4. Descargar datos: 15 min, horario, diario, mensual
5. Múltiples estaciones (región completa)
6. Formato de salida y exportar
7. Gráfico rápido
8. Caché automático

---

## 1. Conexión

```bash
pip install -e .
```

Necesitas una API key de [agromet.inia.cl](https://agromet.inia.cl/api/v2/).

In [ ]:
from iniamet import INIAClient

client = INIAClient(api_key="6c66ed32deb984ce0fb41ffdc54c1e98b517fa50")
print("Conectado ✓")

## 2. Listar estaciones

Se usa el código de región con prefijo **R** (ej: R16 = Ñuble, R08 = Biobío, R07 = Maule).

| Código | Región |
|---|---|
| R15 | Arica y Parinacota |
| R01 | Tarapacá |
| R02 | Antofagasta |
| R03 | Atacama |
| R04 | Coquimbo |
| R05 | Valparaíso |
| R13 | Metropolitana |
| R06 | O'Higgins |
| R07 | Maule |
| R16 | Ñuble |
| R08 | Biobío |
| R09 | La Araucanía |
| R14 | Los Ríos |
| R10 | Los Lagos |
| R11 | Aysén |
| R12 | Magallanes |

In [ ]:
# Una sola región
nuble = client.get_stations(region="R16")
print(f"Estaciones en Ñuble (R16): {len(nuble)}")
nuble[['codigo', 'nombre', 'comuna']].head(10)

In [ ]:
# Varias regiones a la vez
nuble_bio = client.get_stations(region=["R16", "R08"])
print(f"Estaciones en Ñuble + Biobío: {len(nuble_bio)}")
nuble_bio.groupby('region')['codigo'].count()

In [ ]:
# Tres regiones: Ñuble + Biobío + Maule
centro_sur = client.get_stations(region=["R16", "R08", "R07"])
print(f"Estaciones centro-sur: {len(centro_sur)}")
centro_sur.groupby('region')['codigo'].count()

In [ ]:
# Filtrar por tipo de estación
inia_nuble = client.get_stations(region="R16", station_type="INIA")
print(f"Estaciones INIA en Ñuble: {len(inia_nuble)}")
inia_nuble[['codigo', 'nombre', 'comuna']]

## 3. Variables disponibles

### Constantes incluidas

| Constante | ID | Variable |
|---|---|---|
| `VAR_TEMPERATURA_MEDIA` | 2002 | Temperatura del aire (°C) |
| `VAR_PRECIPITACION` | 2001 | Precipitación (mm) |
| `VAR_HUMEDAD_RELATIVA` | 2007 | Humedad relativa (%) |
| `VAR_RADIACION_MEDIA` | 2022 | Radiación solar (W/m²) |
| `VAR_VIENTO_VELOCIDAD_MEDIA` | 2013 | Velocidad viento (m/s) |
| `VAR_VIENTO_VELOCIDAD_MAXIMA` | 2014 | Ráfaga máxima (m/s) |
| `VAR_VIENTO_DIRECCION` | 2012 | Dirección viento (°) |
| `VAR_PRESION_ATMOSFERICA` | 2125 | Presión (mbar) |

In [ ]:
# Ver qué variables tiene una estación específica
variables = client.get_variables("INIA-47")  # Ninhue, Ñuble
variables

In [ ]:
# Tabla completa de variables conocidas
from iniamet import list_all_variables
list_all_variables()

## 4. Descargar datos de UNA estación

### 4a. Datos crudos (cada 15 minutos)

In [ ]:
from iniamet import VAR_TEMPERATURA_MEDIA, VAR_PRECIPITACION

# Temperatura cada 15 min
temp_15min = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-31")

print(f"Registros: {len(temp_15min)}")
print(f"Columnas:  {list(temp_15min.columns)}")
temp_15min.head()

### 4b. Agregación horaria

In [ ]:
temp_horario = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-31", aggregation="horario")
print(f"Registros horarios: {len(temp_horario)}")
temp_horario.head()

### 4c. Agregación diaria

Para temperatura calcula automáticamente **min, max y media**:

In [ ]:
temp_diario = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-31", aggregation="diario")
print(f"Días: {len(temp_diario)}, Columnas: {list(temp_diario.columns)}")
temp_diario.head()

### 4d. Precipitación (se suma, no se promedia)

In [ ]:
pp_diario = client.get_data("INIA-47", VAR_PRECIPITACION, "2025-06-01", "2025-06-30", aggregation="diario")
print(f"Días: {len(pp_diario)}")
pp_diario.head()

### 4e. Agregación mensual o semanal

In [ ]:
temp_mensual = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-12-31", aggregation="mensual")
temp_mensual

### Resumen de agregaciones

| Código | Resolución | También acepta |
|---|---|---|
| (nada) | 15 minutos (crudo) | `"raw"` |
| `"horario"` | Horario | `"hourly"`, `"H"` |
| `"diario"` | Diario | `"daily"`, `"D"` |
| `"semanal"` | Semanal | `"weekly"`, `"W"` |
| `"mensual"` | Mensual | `"monthly"`, `"M"` |

## 5. Descargar MÚLTIPLES estaciones

### 5a. Varias estaciones específicas

In [ ]:
from iniamet import VAR_HUMEDAD_RELATIVA

datos = client.bulk_download(
    stations=["INIA-47", "INIA-139", "INIA-351"],
    variables=[VAR_TEMPERATURA_MEDIA, VAR_HUMEDAD_RELATIVA],
    start_date="2025-01-01",
    end_date="2025-01-31"
)

for key, df in datos.items():
    print(f"{key}: {len(df)} registros")

### 5b. Todas las estaciones de una región

In [ ]:
# Obtener todas las estaciones de Ñuble
est_nuble = client.get_stations(region="R16")
codigos = est_nuble['codigo'].tolist()
print(f"Estaciones R16: {len(codigos)}")

# Descargar temperatura de todas (1 semana de ejemplo)
datos_region = client.bulk_download(
    stations=codigos,
    variables=[VAR_TEMPERATURA_MEDIA],
    start_date="2025-01-01",
    end_date="2025-01-07"
)

print(f"Estaciones con datos: {len(datos_region)}")
for key, df in list(datos_region.items())[:5]:
    print(f"  {key}: {len(df)} registros")

### 5c. Varias regiones a la vez

In [ ]:
# Ñuble + Biobío
est_multi = client.get_stations(region=["R16", "R08"])
codigos_multi = est_multi['codigo'].tolist()
print(f"Estaciones R16+R08: {len(codigos_multi)}")

# Descargar solo las primeras 5 como demo
datos_multi = client.bulk_download(
    stations=codigos_multi[:5],
    variables=[VAR_TEMPERATURA_MEDIA],
    start_date="2025-01-01",
    end_date="2025-01-07"
)

for key, df in datos_multi.items():
    print(f"{key}: {len(df)} registros")

## 6. Formato de salida y exportar

Los datos siempre se devuelven como **pandas DataFrame**.

In [ ]:
df = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-07")

print("Tipo:    ", type(df).__name__)
print("Columnas:", list(df.columns))
print("Dtypes:")
print(df.dtypes)
print(f"\nFilas: {len(df)}")
df.head()

In [ ]:
# Exportar
df.to_csv("temperatura_inia47.csv", index=False)
# df.to_excel("temperatura_inia47.xlsx", index=False)
print("Archivo guardado ✓")

## 7. Gráfico rápido

In [ ]:
import matplotlib.pyplot as plt

td = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-03-31", aggregation="D")

plt.figure(figsize=(12, 4))
plt.fill_between(td['tiempo'], td['valor_min'], td['valor_max'], alpha=0.3, label='Min-Max')
plt.plot(td['tiempo'], td['valor'], 'r-', lw=1, label='Media')
plt.ylabel('Temperatura (°C)')
plt.title('INIA-47 Ninhue - Temperatura diaria (ene-mar 2025)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Caché automático

La librería guarda los datos en `iniamet_cache/` automáticamente.  
La segunda vez que pides los mismos datos, los lee del disco (instantáneo).

- Desactivar: `INIAClient(cache=False)`
- Limpiar: `client.cache_manager.clear_cache()`
- Cambiar ruta: `INIAClient(cache_dir="mi_cache")`

In [ ]:
import time

t0 = time.time()
df1 = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-02-01", "2025-02-28")
print(f"Primera descarga: {time.time()-t0:.1f}s ({len(df1)} registros)")

t0 = time.time()
df2 = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-02-01", "2025-02-28")
print(f"Desde caché:      {time.time()-t0:.3f}s ({len(df2)} registros)")

---
## Resumen rápido

```python
from iniamet import INIAClient, VAR_TEMPERATURA_MEDIA, VAR_PRECIPITACION

client = INIAClient(api_key="...")

# Estaciones
client.get_stations(region="R16")              # una región
client.get_stations(region=["R16", "R08"])     # varias regiones

# Variables de una estación
client.get_variables("INIA-47")

# Descargar datos
client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-12-31")
client.get_data(..., aggregation="D")          # D=diario, H=horario, W=semanal, M=mensual

# Descarga masiva
client.bulk_download(stations=[...], variables=[...], start_date=..., end_date=...)
```

# Tutorial INIAMET - Datos Agroclimáticos de Chile

Esta librería permite acceder a los datos de estaciones agrometeorológicas de INIA Chile.

**Lo que aprenderás:**
1. Conectarse a la API
2. Listar estaciones disponibles
3. Ver variables de una estación
4. Descargar datos (15 min, horario, diario)
5. Descargar múltiples estaciones de una región

---

## 1. Instalación y conexión

Primero, instala la librería (si no lo has hecho):
```bash
pip install -e .
```

Para conectarse necesitas una **API key** de [agromet.inia.cl](https://agromet.inia.cl/api/v2/).

In [ ]:
from iniamet import INIAClient

# Opción 1: pasar la key directamente
client = INIAClient(api_key="TU_API_KEY_AQUI")

# Opción 2: si ya configuraste la variable de entorno INIA_API_KEY
# client = INIAClient()

print("Conectado ✓")

## 2. Listar estaciones

Puedes obtener **todas** las estaciones o filtrar por región.

In [ ]:
# Todas las estaciones del país
todas = client.get_stations()
print(f"Total estaciones en Chile: {len(todas)}")
todas.head()

In [ ]:
# Filtrar por región (acepta nombre, código o número)
nuble = client.get_stations(region="Ñuble")     # por nombre
# nuble = client.get_stations(region="R16")     # por código

print(f"Estaciones en Ñuble: {len(nuble)}")
nuble[['codigo', 'nombre', 'comuna']].head(10)

In [ ]:
# Filtrar por tipo de estación
inia_only = client.get_stations(station_type="INIA")
print(f"Estaciones INIA: {len(inia_only)}")
inia_only[['codigo', 'nombre', 'region']].head()

## 3. Variables disponibles

Cada estación tiene distintas variables. Veamos cuáles tiene una estación:

In [ ]:
variables = client.get_variables("INIA-47")  # Ninhue, Ñuble
variables

### Variables comunes (constantes)

Para no tener que recordar los IDs numéricos, la librería trae constantes:

| Constante | ID | Variable |
|---|---|---|
| `VAR_TEMPERATURA_MEDIA` | 2002 | Temperatura del aire (°C) |
| `VAR_PRECIPITACION` | 2001 | Precipitación (mm) |
| `VAR_HUMEDAD_RELATIVA` | 2007 | Humedad relativa (%) |
| `VAR_RADIACION_MEDIA` | 2022 | Radiación solar (W/m²) |
| `VAR_VIENTO_VELOCIDAD_MEDIA` | 2013 | Velocidad viento (m/s) |
| `VAR_VIENTO_VELOCIDAD_MAXIMA` | 2014 | Ráfaga máxima (m/s) |
| `VAR_VIENTO_DIRECCION` | 2012 | Dirección viento (°) |
| `VAR_PRESION_ATMOSFERICA` | 2125 | Presión (mbar) |

In [ ]:
# También puedes listar todas las variables conocidas
from iniamet import list_all_variables
list_all_variables()

## 4. Descargar datos de UNA estación

### 4a. Datos crudos (cada 15 minutos)

In [ ]:
from iniamet import VAR_TEMPERATURA_MEDIA, VAR_PRECIPITACION

# Temperatura cada 15 minutos
temp_raw = client.get_data(
    station="INIA-47",
    variable=VAR_TEMPERATURA_MEDIA,
    start_date="2025-01-01",
    end_date="2025-01-31"
)

print(f"Registros: {len(temp_raw)}")
print(f"Columnas: {list(temp_raw.columns)}")
temp_raw.head()

### 4b. Agregación horaria

In [ ]:
# Temperatura promedio por hora
temp_horario = client.get_data(
    station="INIA-47",
    variable=VAR_TEMPERATURA_MEDIA,
    start_date="2025-01-01",
    end_date="2025-01-31",
    aggregation="H"  # H=horario
)

print(f"Registros horarios: {len(temp_horario)}")
temp_horario.head()

### 4c. Agregación diaria

Para temperatura, automáticamente calcula **min, max y media**:

In [ ]:
# Temperatura diaria (incluye min, max, media automáticamente)
temp_diario = client.get_data(
    station="INIA-47",
    variable=VAR_TEMPERATURA_MEDIA,
    start_date="2025-01-01",
    end_date="2025-01-31",
    aggregation="D"  # D=diario
)

print(f"Días: {len(temp_diario)}")
print(f"Columnas: {list(temp_diario.columns)}")
temp_diario.head()

### 4d. Precipitación (se suma en vez de promediar)

In [ ]:
# Precipitación diaria acumulada
pp_diario = client.get_data(
    station="INIA-47",
    variable=VAR_PRECIPITACION,
    start_date="2025-06-01",
    end_date="2025-06-30",
    aggregation="D"
)

print(f"Días con datos: {len(pp_diario)}")
pp_diario.head()

### 4e. Agregación semanal o mensual

In [ ]:
# Temperatura media mensual
temp_mensual = client.get_data(
    station="INIA-47",
    variable=VAR_TEMPERATURA_MEDIA,
    start_date="2025-01-01",
    end_date="2025-12-31",
    aggregation="M"  # M=mensual, W=semanal
)

temp_mensual

## 5. Descargar MÚLTIPLES estaciones

### 5a. Descarga masiva con `bulk_download`

In [ ]:
from iniamet import VAR_HUMEDAD_RELATIVA

# Descargar temperatura y humedad de 3 estaciones
datos = client.bulk_download(
    stations=["INIA-47", "INIA-139", "INIA-351"],
    variables=[VAR_TEMPERATURA_MEDIA, VAR_HUMEDAD_RELATIVA],
    start_date="2025-01-01",
    end_date="2025-01-31"
)

# Resultado: diccionario {"estacion_variable": DataFrame}
for key, df in datos.items():
    print(f"{key}: {len(df)} registros")

### 5b. Todas las estaciones de una región

In [ ]:
# Obtener códigos de todas las estaciones de Ñuble
estaciones_nuble = client.get_stations(region="Ñuble")
codigos = estaciones_nuble['codigo'].tolist()
print(f"Descargando temperatura de {len(codigos)} estaciones...")

# Descargar temperatura de todas
datos_region = client.bulk_download(
    stations=codigos,
    variables=[VAR_TEMPERATURA_MEDIA],
    start_date="2025-01-01",
    end_date="2025-01-07"  # una semana para este ejemplo
)

print(f"\nEstaciones con datos: {len(datos_region)}")
for key, df in datos_region.items():
    print(f"  {key}: {len(df)} registros")

## 6. Formato de salida

Todos los datos se devuelven como **pandas DataFrame**. Puedes exportarlos fácilmente:

In [ ]:
# El formato siempre es un DataFrame con columnas 'tiempo' y 'valor'
temp_raw = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-07")

print("Tipo:", type(temp_raw))
print("Columnas:", list(temp_raw.columns))
print("Dtypes:")
print(temp_raw.dtypes)
print()
temp_raw.head()

In [ ]:
# Exportar a CSV
temp_raw.to_csv("temperatura_inia47.csv", index=False)

# Exportar a Excel
# temp_raw.to_excel("temperatura_inia47.xlsx", index=False)

print("Archivo guardado ✓")

## 7. Gráfico rápido

In [ ]:
import matplotlib.pyplot as plt

temp_diario = client.get_data(
    "INIA-47", VAR_TEMPERATURA_MEDIA,
    "2025-01-01", "2025-03-31",
    aggregation="diario"
)

plt.figure(figsize=(12, 4))
plt.fill_between(temp_diario['tiempo'], temp_diario['valor_min'], temp_diario['valor_max'], alpha=0.3, label='Min-Max')
plt.plot(temp_diario['tiempo'], temp_diario['valor'], 'r-', linewidth=1, label='Media')
plt.ylabel('Temperatura (°C)')
plt.title('INIA-47 Ninhue - Temperatura diaria')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Caché automático

La librería guarda automáticamente los datos descargados en una carpeta `iniamet_cache/`.
Si pides los mismos datos otra vez, los lee del caché (mucho más rápido).

- El caché se crea automáticamente
- Para desactivarlo: `INIAClient(cache=False)`
- Para limpiarlo: `client.cache_manager.clear_cache()`
- Para cambiar la ubicación: `INIAClient(cache_dir="mi_carpeta")`

In [ ]:
# Primera vez: descarga de la API (lento)
# Segunda vez: lee del caché (instantáneo)
import time

t0 = time.time()
df1 = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-31")
t1 = time.time()
print(f"Primera descarga: {t1-t0:.1f}s ({len(df1)} registros)")

t0 = time.time()
df2 = client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-01-31")
t1 = time.time()
print(f"Desde caché:      {t1-t0:.3f}s ({len(df2)} registros)")

## Resumen de la API

```python
from iniamet import INIAClient, VAR_TEMPERATURA_MEDIA, VAR_PRECIPITACION

client = INIAClient(api_key="...")       # Conectar
client.get_stations(region="R16")         # Listar estaciones (Ñuble)
client.get_stations(region=["R16","R08"]) # Varias regiones
client.get_variables("INIA-47")           # Variables de una estación

# Descargar datos (una estación, una variable)
client.get_data("INIA-47", VAR_TEMPERATURA_MEDIA, "2025-01-01", "2025-12-31")
client.get_data(..., aggregation="diario")  # diario, horario, semanal, mensual

# Descargar datos (múltiples estaciones y variables)
client.bulk_download(stations=[...], variables=[...], start_date=..., end_date=...)
```